In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# ==========================================
# 2. CONFIG: LOKASI FILE
# ==========================================
# Path file Ground Truth (Kunci Jawaban)
file_path_gt = '/content/drive/MyDrive/DataSet/ground_truth_selesai_1000.csv'

# Path file Berita (Dataset Utama)
file_path_berita = '/content/drive/MyDrive/DataSet/indonesia_news_1000.csv'

df.head(10)

,Query_ID,Query_Text,Rank_BM25,Judul_Berita,Isi_Singkat,Relevan
0,1,kenaikan harga bahan pokok,1,PM Anwar Ibrahim Umumkan Bantuan Tunai dan Tur...,"Baca berita dengan sedikit iklan, klik di sin...",0
1,1,kenaikan harga bahan pokok,2,Pertamina Klaim Tak Cari Untung dari Kelangkaa...,"Baca berita dengan sedikit iklan, klik di sin...",0
2,1,kenaikan harga bahan pokok,3,"PPN 12 Persen Hanya untuk Barang Mewah, Dinila...","TANGERANG, KOMPAS.com - Batalnya kenaikan Paj...",0
3,1,kenaikan harga bahan pokok,4,Menteri Amran Bantah Beras Mahal Akibat Bulog ...,"Baca berita dengan sedikit iklan, klik di sin...",1
4,1,kenaikan harga bahan pokok,5,KAI Bantah Harga Tiket Kereta Api Naik Usai Le...,"JAKARTA, KOMPAS.com - PT Kereta Api Indonesia...",0
5,1,kenaikan harga bahan pokok,6,Kementerian Aparatur Negara: Belum Ada Pembaha...,"Baca berita dengan sedikit iklan, klik di sin...",0
6,1,kenaikan harga bahan pokok,7,Kata Pramono Anung Soal Kenaikan Tarif PAM Jay...,"Baca berita dengan sedikit iklan, klik di sin...",1
7,1,kenaikan harga bahan pokok,8,Prediksi Harga Emas Sepekan ke Depan,"Baca berita dengan sedikit iklan, klik di sin...",0
8,1,kenaikan harga bahan pokok,9,PBB: Israel Bunuh 1.054 Warga Gaza Saat Antre ...,Perserikatan Bangsa-Bangsa (PBB) menyebut mili...,0
9,1,kenaikan harga bahan pokok,10,Badai Buntut Perang Iran-Israel,"Baca berita dengan sedikit iklan, klik di sin...",1


In [ ]:
# ==========================================
# CELL 3: PELATIHAN MODEL (TF-IDF & BM25)
# ==========================================
print("⚙️ Sedang Melatih Model...")

# 1. Siapkan Stemmer
factory = StemmerFactory()
stemmer_query = factory.create_stemmer()

# 2. Latih TF-IDF
print("   - Melatih TF-IDF...")
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df_berita['Cleaned_Content'])

# 3. Latih BM25
print("   - Melatih BM25...")
tokenized_corpus = [doc.split(" ") for doc in df_berita['Cleaned_Content']]
bm25 = BM25Okapi(tokenized_corpus)

print("✅ Model Siap Digunakan!")

⚙️ Sedang Melatih Model...
   - Melatih TF-IDF...
   - Melatih BM25...
✅ Model Siap Digunakan!


In [ ]:
# ==========================================
# CELL 4: FUNGSI HITUNG METRIK
# ==========================================
def calculate_metrics_at_k(retrieved_docs, relevant_docs, k_list):
    metrics = {}
    total_relevant = len(relevant_docs)

    # Jika tidak ada kunci jawaban yang relevan
    if total_relevant == 0:
        return {k: {'P': 0, 'R': 0, 'F1': 0} for k in k_list}, 0

    # --- Hitung Average Precision (AP) ---
    precisions_for_ap = []
    rel_count = 0
    for i, doc in enumerate(retrieved_docs, 1):
        if doc in relevant_docs:
            rel_count += 1
            precisions_for_ap.append(rel_count / i)

    ap = sum(precisions_for_ap) / total_relevant if total_relevant > 0 else 0

    # --- Hitung P, R, F1 per K ---
    for k in k_list:
        retrieved_k = retrieved_docs[:k]
        rel_k = len([d for d in retrieved_k if d in relevant_docs])

        p = rel_k / k
        r = rel_k / total_relevant
        f1 = 2*(p*r)/(p+r) if (p+r) > 0 else 0

        metrics[k] = {'P': p, 'R': r, 'F1': f1}

    return metrics, ap

print("✅ Rumus evaluasi siap.")

✅ Rumus evaluasi siap.


In [ ]:
# ==========================================
# CELL 5: EKSEKUSI EVALUASI
# ==========================================
file_path_gt = '/content/drive/MyDrive/DataSet/ground_truth_selesai_1000.csv'
k_values = [5, 10, 20] # Evaluasi pada k=5, 10, dan 20

print("🚀 Memulai Evaluasi Sistem...")

# 1. Load Ground Truth
try:
    df_gt = pd.read_csv(file_path_gt, sep=';') # Coba titik koma
    if len(df_gt.columns) < 2:
        df_gt = pd.read_csv(file_path_gt, sep=',') # Fallback ke koma
    print(f"✅ Ground Truth dimuat: {len(df_gt)} baris.")
except Exception as e:
    raise FileNotFoundError(f"❌ Gagal baca file GT: {e}")

hasil_evaluasi = []
queries = df_gt['Query_Text'].unique()

# 2. Looping per Query
for q in queries:
    target_docs = df_gt[(df_gt['Query_Text'] == q) & (df_gt['Relevan'] == 1)]['Judul_Berita'].tolist()
    if len(target_docs) == 0: continue

    # Preprocessing Query
    try: q_stem = stemmer_query.stem(q.lower())
    except: q_stem = q.lower()

    # --- TF-IDF ---
    q_vec = vectorizer.transform([q_stem])
    scores_tfidf = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_idx_tfidf = scores_tfidf.argsort()[-20:][::-1]
    hasil_tfidf = [df_berita.iloc[i]['Judul'] for i in top_idx_tfidf]
    m_tfidf, ap_tfidf = calculate_metrics_at_k(hasil_tfidf, target_docs, k_values)

    # --- BM25 ---
    scores_bm25 = bm25.get_scores(q_stem.split(" "))
    top_idx_bm25 = np.argsort(scores_bm25)[-20:][::-1]
    hasil_bm25 = [df_berita.iloc[i]['Judul'] for i in top_idx_bm25]
    m_bm25, ap_bm25 = calculate_metrics_at_k(hasil_bm25, target_docs, k_values)

    # Simpan Hasil Lengkap
    row_t = {'Model': 'TF-IDF', 'Query': q, 'AP': ap_tfidf}
    row_b = {'Model': 'BM25',  'Query': q, 'AP': ap_bm25}

    for k in k_values:
        # Simpan P, R, F1 untuk TF-IDF
        row_t[f'P@{k}'] = m_tfidf[k]['P']; row_t[f'R@{k}'] = m_tfidf[k]['R']; row_t[f'F1@{k}'] = m_tfidf[k]['F1']
        # Simpan P, R, F1 untuk BM25
        row_b[f'P@{k}'] = m_bm25[k]['P']; row_b[f'R@{k}'] = m_bm25[k]['R']; row_b[f'F1@{k}'] = m_bm25[k]['F1']

    hasil_evaluasi.append(row_t)
    hasil_evaluasi.append(row_b)

print("✅ Evaluasi Selesai!")

🚀 Memulai Evaluasi Sistem...
✅ Ground Truth dimuat: 100 baris.
✅ Evaluasi Selesai!


In [ ]:
# ==========================================
# CELL 6: OUTPUT HASIL (TABEL SKOR)
# ==========================================
if hasil_evaluasi:
    df_hasil = pd.DataFrame(hasil_evaluasi)
    summary = df_hasil.groupby('Model').mean(numeric_only=True)

    print("\n" + "="*100)
    print("📊 TABEL PERBANDINGAN PERFORMA FINAL (Lengkap)")
    print("="*100)

    # Menampilkan kolom yang diminta tugas: AP (MAP), P, R, F1
    cols = ['AP', 'P@5', 'R@5', 'F1@5', 'P@10', 'R@10', 'F1@10', 'P@20', 'R@20', 'F1@20']
    print(summary[cols].round(4))

    print("="*100)
    print("KETERANGAN:")
    print("AP   = MAP (Mean Average Precision)")
    print("P@k  = Precision pada k")
    print("R@k  = Recall pada k")
    print("F1@k = F-Measure pada k")

    # Simpan detailnya
    df_hasil.to_csv('laporan_akhir_ir.csv', index=False)
    print("✅ File detail tersimpan: laporan_akhir_ir.csv")
else:
    print("⚠️ Tidak ada hasil. Cek apakah ada angka '1' di file Ground Truth.")


📊 TABEL PERBANDINGAN PERFORMA FINAL (Lengkap)
            AP   P@5     R@5    F1@5  P@10    R@10   F1@10   P@20  R@20  \
Model                                                                     
BM25    0.7559  0.58  0.5164  0.5259  0.57  1.0000  0.7026  0.285   1.0   
TF-IDF  0.6976  0.58  0.5147  0.5284  0.48  0.8333  0.5899  0.285   1.0   

         F1@20  
Model           
BM25    0.4331  
TF-IDF  0.4331  
KETERANGAN:
AP   = MAP (Mean Average Precision)
P@k  = Precision pada k
R@k  = Recall pada k
F1@k = F-Measure pada k
✅ File detail tersimpan: laporan_akhir_ir.csv
